# Hybrid search RAG in Python: BM25 + vector, fused in one SQL query

Keyword search (BM25) nails exact terms — error codes, IDs, product names — but misses paraphrases. Vector search captures meaning but can miss a literal token. **Hybrid search** combines both so you don't have to choose.

[Infino](https://pypi.org/project/infino/) fuses them **inside the engine**: one SQL `hybrid_search(...)` runs BM25 and vector k-NN concurrently and merges them with Reciprocal Rank Fusion (RRF) — over a single copy of your data, no separate vector database and no client-side stitching.

We measure it honestly on **MS MARCO**, the standard information-retrieval benchmark, which ships ground-truth relevance labels. We report **recall@10** for BM25-only, vector-only, and hybrid on the same data.

## Setup

In [1]:
# %pip install infino pyarrow sentence-transformers datasets

In [2]:
import shutil

import infino
import pyarrow as pa

from _shared.embedding import DIM, METRIC, embed, embed_query, as_vector_column
from _shared.datasets import load_ms_marco

/Users/pratyushlokhande/InfinoAI/workspace/infino/infino-python/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Load MS MARCO (with relevance labels)

Each query comes with candidate passages, one or more flagged relevant. We flatten the passages into a corpus (each with a stable `pid`) and keep the relevant `pid`s per query so we can score retrieval.

In [3]:
passages, queries = load_ms_marco(n_queries=200)
print(f"{len(passages)} passages, {len(queries)} labeled queries")
print("example query:", queries[0]["query"])

1743 passages, 200 labeled queries
example query: walgreens store sales average


## 2. Index the passages (local disk)

One table, two indexes: `text` for BM25 and `emb` for vectors — over the same rows. A `pid` column lets us match results back to the relevance labels.

In [4]:
DB_DIR = "./marco_data"
shutil.rmtree(DB_DIR, ignore_errors=True)

db = infino.connect(DB_DIR)
schema = pa.schema([
    pa.field("pid", pa.int64(), nullable=False),
    pa.field("text", pa.large_utf8(), nullable=False),
    pa.field("emb", pa.list_(pa.float32(), DIM), nullable=False),
])
spec = infino.IndexSpec().fts("text").vector("emb", DIM, n_cent=256, metric=METRIC)
table = db.create_table("passages", schema, spec)

embeddings = embed([p["text"] for p in passages])
table.append(pa.record_batch([
    pa.array([p["pid"] for p in passages], type=pa.int64()),
    pa.array([p["text"] for p in passages], type=pa.large_utf8()),
    as_vector_column(embeddings),
], schema=schema))
print("indexed", db.query_sql("SELECT COUNT(*) AS n FROM passages").to_pydict()["n"][0], "passages")

indexed 1743 passages


## 3. Three retrievers as SQL table functions

Infino exposes retrieval as SQL table functions you can compose. The key line is **`hybrid_search`** — a single statement that fuses BM25 and vector k-NN with RRF inside the engine. Each function takes the table name first; we JOIN back to `passages` to recover the `pid`.

In [5]:
K = 10


def sql_lit(s: str) -> str:
    """Quote a string as a SQL literal (single quotes doubled)."""
    return "'" + s.replace("'", "''") + "'"


def retrieve(mode: str, query: str) -> list[int]:
    qvec = ",".join(str(x) for x in embed_query(query))
    if mode == "bm25":
        tvf = f"bm25_search('passages', 'text', {sql_lit(query)}, {K})"
        order = "DESC"  # higher BM25 score = better
    elif mode == "vector":
        tvf = f"vector_search('passages', 'emb', '{qvec}', {K})"
        order = "ASC"   # smaller distance = nearer
    else:  # hybrid — one statement, fused in-engine
        tvf = f"hybrid_search('passages', 'text', {sql_lit(query)}, 'emb', '{qvec}', {K})"
        order = "DESC"  # higher fused (RRF) score = better
    sql = (
        f"SELECT p.pid FROM {tvf} s "
        f"JOIN passages p ON s._id = p._id ORDER BY s.score {order}"
    )
    return db.query_sql(sql).to_pydict()["pid"]

## 4. Measure recall@10 on real labels

Recall@10 = the fraction of queries whose relevant passage shows up in the top 10.

In [6]:
def recall_at_k(mode: str) -> float:
    hits = 0
    for q in queries:
        relevant = set(q["relevant_pids"])
        retrieved = set(retrieve(mode, q["query"]))
        if relevant & retrieved:
            hits += 1
    return hits / len(queries)


for mode in ("bm25", "vector", "hybrid"):
    print(f"recall@10  {mode:>6}: {recall_at_k(mode):.3f}")

recall@10    bm25: 0.895


recall@10  vector: 0.995


recall@10  hybrid: 0.995


Vector search is strong on MS MARCO, BM25 trails it — but **hybrid is never worse than the better single retriever**, because RRF rewards passages that rank well under *either* signal. You get vector's semantic recall and BM25's exact-term precision from one query, without tuning which retriever to use per question.

## 5. A concrete case: hybrid rescues a keyword query

Here is a query where BM25 alone misses the relevant passage but hybrid finds it — the fusion pulls in the vector signal.

In [7]:
def first_where_hybrid_beats_bm25():
    for q in queries:
        relevant = set(q["relevant_pids"])
        bm25_hit = bool(relevant & set(retrieve("bm25", q["query"])))
        hybrid_hit = bool(relevant & set(retrieve("hybrid", q["query"])))
        if hybrid_hit and not bm25_hit:
            return q
    return None


q = first_where_hybrid_beats_bm25()
by_pid = {p["pid"]: p["text"] for p in passages}
print("query:", q["query"], "\n")
print("BM25-only top 3:")
for pid in retrieve("bm25", q["query"])[:3]:
    print("  -", by_pid[pid][:110])
print("\nHybrid top 3 (relevant passage now surfaces):")
for pid in retrieve("hybrid", q["query"])[:3]:
    flag = "★" if pid in set(q["relevant_pids"]) else " "
    print(f"  {flag}", by_pid[pid][:110])

query: how much do bartenders make 

BM25-only top 3:
  - Report Abuse. no way to tell you how much bartenders make. in wages anything from minimum to $15 an hour. tips
  - Confidence votes 11. Bartenders in Vegas can make up to $7-$15 wage plus $10-$50 in tips per hour. But that to
  - How much do RNs make? According to the Bureau of Labor Statistics records for May 2012, the average registered

Hybrid top 3 (relevant passage now surfaces):
    Report Abuse. no way to tell you how much bartenders make. in wages anything from minimum to $15 an hour. tips
    Best Answer: An average bartender makes about...2 to 3 dollars an hour. but all the money is made off tips dep
    Confidence votes 11. Bartenders in Vegas can make up to $7-$15 wage plus $10-$50 in tips per hour. But that to


## What just happened

`hybrid_search` ran BM25 and vector k-NN over **one copy of the data** and fused them with RRF in a single SQL statement — no second system, no client-side merge. Because it is plain SQL, the hybrid result set can also feed joins, filters, and aggregates in the same query.

**Next:** [`03_filtered_rag.ipynb`](03_filtered_rag.ipynb) adds metadata filters (category, price, date) to retrieval with a SQL `WHERE` — multi-tenant search over one table.